In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Edits
import matplotlib as mpl
from mpl_toolkits.mplot3d import Axes3D

from scipy.integrate import odeint
from scipy.optimize import fsolve
import scipy.constants as cts
import pylcp
import pylcp.atom as atom
from pylcp.fields import conventional3DMOTBeams
from pylcp.common import bisectFindChangeValue, progressBar
#plt.style.use('paper')

import time

Run this cell for units where $x_0=\hbar\Gamma/\mu_B B'$ and $t_0 = k x_0/\Gamma$:

In [ ]:
# 50 gauss/cm for Sr 
# mub on database
# 0.43 cm for x0 30/1.4/50

# x is length unit
# k is wavenumber which relates to wavelength
x0 = 0.1 # cm
k = 2*np.pi/461E-7 # cm^{-1}
kbar = k*x0

# D source for position
d = np.array([0., -8.839, -8.839])

# Gamma is decay rate
# t0 is normalized time units of decary
# wb is width factor
gamma = 2*np.pi*30e6
t0 = 1e-5 # s
gammabar = gamma*t0
alpha = 2*np.pi*cts.value('Bohr magneton in Hz/T')*50*1e-4*x0*t0

mbar = 87.9056122571*cts.value('atomic mass constant')*(x0*1e-2)**2/(cts.hbar*t0)

wb = 4.7 # mm
print(x0, t0, k, kbar, gamma, gammabar, mbar, wb, alpha, gammabar/kbar)

Run this cell for units where $x_0=1/k$ and $t_0 = 1/\Gamma$

In [ ]:
# k = 2*np.pi/461E-7 # cm^{-1}
# x0 = 1/k # cm

# gamma = 2*np.pi*30e6
# t0 = 1/gamma

# # D source for position
# d = np.array([0., -8.839/10, -8.839/10])/x0

# # Gamma is decay rate
# # t0 is normalized time units of decary
# # wb is width factor

# kbar = 1
# gammabar = 1
# mbar = 87.9056122571*cts.value('atomic mass constant')*x0**2/(cts.hbar*t0)
# alpha = 2*np.pi*cts.value('Bohr magneton in Hz/T')*50*1e-4*x0*t0

# wb = .47/x0
# print(x0, t0, k, kbar, gamma, gammabar, mbar, wb, alpha)

### First, we define the simple problem:

In [ ]:
class aPHIMOT(pylcp.laserBeams):
     def __init__(self, *args, **kwargs):
        super().__init__()
    
        beam_type = kwargs.pop('beam_type', pylcp.laserBeam)
        pol = kwargs.pop('pol', +1)
        kmag = kwargs.pop('k', 1.)
        
        self.add_laser(beam_type(kmag*(np.array([0., 0.,  17.678])/(np.linalg.norm(np.array([0., 0.,  17.678])))), +pol, *args, **kwargs))
        self.add_laser(beam_type(kmag*(np.array([0., 0.,  -17.678])/(np.linalg.norm(np.array([0., 0.,  -17.678])))), +pol, *args, **kwargs))
        self.add_laser(beam_type(kmag*(np.array([10.825, -13.258, 4.419])/(np.linalg.norm(np.array([10.825, -13.258, 4.419])))), -pol, *args, **kwargs))
        self.add_laser(beam_type(kmag*(np.array([10.825, 13.258, -4.419])/(np.linalg.norm(np.array([10.825, 13.258, -4.419])))), -pol, *args, **kwargs))
        self.add_laser(beam_type(kmag*(np.array([-10.825, -13.258, 4.419])/(np.linalg.norm(np.array([-10.825, -13.258, 4.419])))), -pol, *args, **kwargs))
        self.add_laser(beam_type(kmag*(np.array([-10.825, 13.258, -4.419])/(np.linalg.norm(np.array([-10.825, 13.258, -4.419])))), -pol, *args, **kwargs))
    

# Set up the laser beams with their appropriate characteristics
# det is detuning of frequency
# alpha relates to magnetic field gradient
# beta is normalized intensity
det = -1.5*gammabar
beta = 1.0

#beam_to_sim = pylcp.infinitePlaneWaveBeam
#beam_to_sim = pylcp.gaussianBeam
beam_to_sim = pylcp.clippedGaussianBeam

#MOT_to_sim = conventional3DMOTBeams
MOT_to_sim = aPHIMOT

#MOT_to_sim_kwargs = {'rotation_angles':[np.pi/4, 0., 0.]} # Extra arguments for conventional3DMOTBeams
MOT_to_sim_kwargs = {} # Extra arguments for aPHIMOT

#laser_kwargs = {}
#laser_kwargs = {'wb':wb} # Extra arguments for GaussianBeam
laser_kwargs = {'wb':1000*wb, 'rs':wb} # Extra arguments for clippedGaussianBeam

laserBeams = MOT_to_sim(beta=beta, delta=det, beam_type=beam_to_sim, k=kbar, **laser_kwargs, **MOT_to_sim_kwargs) # kbar

#print(laserBeams.pol())

if (np.sum(np.isnan(laserBeams.pol()), axis=(0, 1)) > 0):
    del laserBeams
    laserBeams = MOT_to_sim(beta=beta, delta=det, beam_type=beam_to_sim, k=kbar, **laser_kwargs, **MOT_to_sim_kwargs)

magField = pylcp.quadrupoleMagneticField(alpha)

equation = pylcp.heuristiceq(laserBeams, magField, gamma=gammabar, mass=mbar)

# Hg, mugq = pylcp.hamiltonians.singleF(F=0, muB=1)
# He, mueq = pylcp.hamiltonians.singleF(F=1, muB=1)

# dq = pylcp.hamiltonians.dqij_two_bare_hyperfine(0, 1)

# hamiltonian = pylcp.hamiltonian(mass=mbar)
# hamiltonian.add_H_0_block('g', Hg)
# hamiltonian.add_H_0_block('e', He)
# hamiltonian.add_mu_q_block('g', mugq)
# hamiltonian.add_mu_q_block('e', mueq)
# hamiltonian.add_d_q_block('g','e', dq, gamma=gammabar, k=kbar)

# equation = pylcp.rateeq(laserBeams, magField, hamiltonian)

In [ ]:
# x_beta = 1/x0
# X, Y = np.meshgrid(np.linspace(-x_beta, x_beta, 101),
#                    np.linspace(-x_beta, x_beta, 101))
# z_tests = np.array([-0.2, 0, 0.2])/x0 # position

# plt.figure("Laser Beams", figsize=(4, 1.5*6)) # 6 beams
# plt.clf()
# # pr = cProfile.Profile()

# for jj, laserBeam in enumerate(laserBeams.beam_vector):
#     for ii, z_test in enumerate(z_tests):
#         Z = z_test*np.ones(X.shape)
#         Rt=np.array([X, Y, Z])

#         BETA = laserBeam.beta(Rt)

#         plt.subplot(len(laserBeams.beam_vector), len(z_tests), jj*len(z_tests)+ii+1)
#         plt.imshow(BETA, origin='lower',
#                    extent=(-x_beta*x0, x_beta*x0,
#                            -x_beta*x0, x_beta*x0))
#         plt.clim((0, 1))
#         #plt.set_cmap('jet')
#         # Make a cross-hair:
#         plt.plot([0, 0], [-x_beta*x0, x_beta*x0],
#                  'w-', linewidth=0.25)
#         plt.plot([-x_beta*x0, x_beta*x0], [0, 0],
#                  'w-', linewidth=0.25)

### Let's calculate the force along the line between the source and the origin:

In [ ]:
# To run, uncomment and change the shared variable name equation to either rate eq or heuristic

# r = np.linspace(-np.linalg.norm(d), -1e-3, 101)

# heuristic.generate_force_profile(np.array([np.zeros(r.shape), -r/np.sqrt(2), -r/np.sqrt(2)]),
#                                  np.zeros((3,)+ r.shape),
#                                  name='F_along_d')
# rateeq.generate_force_profile(np.array([np.zeros(r.shape), -r/np.sqrt(2), -r/np.sqrt(2)]),
#                                  np.zeros((3,)+ r.shape),
#                                  name='F_along_d')

# fig, ax = plt.subplots(3, 2, figsize=(6.5, 5.75))
# ax[0, 0].plot(r, heuristic.profile['F_along_d'].f['g->e'][0])
# ax[0, 0].plot(r, heuristic.profile['F_along_d'].F[0], 'k-')
# ax[0, 0].plot(r, rateeq.profile['F_along_d'].f['g->e'][0], '--')
# ax[0, 0].plot(r, rateeq.profile['F_along_d'].F[0], 'k--')

# ax[0, 1].plot(r, heuristic.profile['F_along_d'].f['g->e'][0, :, 2]+heuristic.profile['F_along_d'].f['g->e'][0, :, 4])
# ax[0, 1].plot(r, heuristic.profile['F_along_d'].f['g->e'][0, :, 3]+heuristic.profile['F_along_d'].f['g->e'][0, :, 5])

# B = np.zeros((3,) + r.shape)
# pols = np.zeros((laserBeams.num_of_beams, 3) + r.shape)
# betas = np.zeros((laserBeams.num_of_beams,) + r.shape)
                
# for ii, r_i in enumerate(r):
#     r_vec = np.array([0., -r_i/np.sqrt(2), -r_i/np.sqrt(2)])
#     B[:, ii] = magField.Field(r_vec)
#     pols[:, :, ii] = np.abs(laserBeams.project_pol(B[:, ii]/np.linalg.norm(B[:, ii]), r_vec))
#     betas[:, ii] = laserBeams.beta(r_vec)
    
# ax[1, 0].plot(r, B.T)
# syms = ['-', '--', '-.'] 
# for ii in range(2, pols.shape[0]):
#     for jj in range(pols.shape[1]):
#         ax[1, 1].plot(r, pols[ii, jj], syms[jj], color='C%d'%ii)
            
# for ii in range(2, betas.shape[0]):
#     ax[2, 0].plot(r, betas[ii], color='C%d'%ii)

# for ii in range(2, 4):
#     ax[2, 1].plot(r, betas[ii] - betas[ii+2], color='C%d'%ii)
# [print(ii, pols[ii, :, 71], betas[ii, 71]) for ii in range(6)];

In [ ]:
# Testing the polarization

# laserBeams.beam_vector[0].pol()

In [ ]:
# # Testing symmetry

# # fig, ax = plt.subplots(1, 1, figsize=(4, 2.75))

# # print(laserBeams.project_pol(np.array([0, -1, -1])))

# for n in np.arange(-10., 10., 1.):
#     a = laserBeams.project_pol(np.array([0., -1., -1.]), np.array([0, n, n]))
#     for array1 in a:
#         print(np.abs(array1))
#     print("\n")

# print("--------------------------------------------------")

# for num in np.arange(-10., 10., 1.):
#     arr = laserBeams.beta(np.array([0, num, num]))
# #    print(arr)
#     for e in [0,2,4]:
# #         ax.plot(num, arr[e+1] - arr[e], linewidth=0.375)
#         print(arr[e+1] - arr[e])
#     print("\n")

### Now compute the background force profile:

In [ ]:
# Create the forces
#dz = 0.025
#dv = 0.25 # Like pixels, if you increase this it runs faster
z = np.linspace(-10, 10, 101) # mm
v = np.linspace(-1, 1, 101) # mm/10 microsec

dz = np.mean(np.diff(z))
dv = np.mean(np.diff(v))

Z, V = np.meshgrid(z, v)

# Rfull = np.array([np.zeros(Z.shape), np.zeros(Z.shape), Z]) Changed when X = Y = 0
# Vfull = np.array([np.zeros(Z.shape), np.zeros(Z.shape), V])

# Computing force along this line
Rfull = np.array([np.zeros(Z.shape), np.zeros(Z.shape), Z])
Vfull = np.array([np.zeros(Z.shape), np.zeros(Z.shape), V]) 

equation.generate_force_profile(Rfull, Vfull, name='Fz', progress_bar=True)

Plot 'er up:

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4, 2.75))
plt.imshow(equation.profile['Fz'].F[2]/kbar/gammabar, origin='bottom',
           extent=(np.amin(z)-dz/2, np.amax(z)+dz/2,
                   np.amin(v)-dv/2, np.amax(v)+dv/2),
           aspect='auto')
cb1 = plt.colorbar()
cb1.set_label('$F/(\hbar k \Gamma)$')
ax.set_xlabel('$x$ (mm)')
ax.set_ylabel('$v/(\Gamma/k)$')
fig.subplots_adjust(left=0.12,right=0.9)

#fig.savefig() 

### Now, let's add in trajectories in phase space:

The first thing to do is define a stop condition, when the velocity of the atom gets to be too small.

In [ ]:
# Define when the program stops and also describes the trajectories. Sets the initial motion/velocity and then evolves the motion

v0s = np.arange(0.1, 1., 0.1)

# See solve_ivp documentation for event function discussion:
def captured_condition(t, y, threshold=1e-5):
    if np.linalg.norm(y[-6:-3])<threshold  and np.linalg.norm(y[-3:])<0.1:
        val = -1.
    else:
        val = 1.
    
    return val

def lost_condition(t, y, threshold=1e-5):
    if np.linalg.norm(y[-3:])>15.:
        val = -1.
    else:
        val = 1.
    
    return val

def nan_found(t, y):
    if np.sum(np.isnan(y)) > 0:
        return -1.
    else:
        return 1.

captured_condition.terminal=True
lost_condition.terminal=True
nan_found.terminal=True

sols = []
for v0 in v0s:
    equation.set_initial_position_and_velocity(d, -1*v0*d/np.linalg.norm(d))
#    equation.set_initial_position_and_velocity(np.array([0,0,z[0]]), np.array([0,0,v0]))
#    equation.set_initial_pop(np.array([1.,0,0,0]))
    equation.evolve_motion([0., 1000.], events=[captured_condition, lost_condition, nan_found], max_step=1., progress_bar=True)
    sols.append(equation.sol)

Now, plot it up:

In [ ]:
# Phase diagrams are only useful right now in certain axises
fig, ax = plt.subplots(1, 1, figsize=(4, 2.75))
plt.imshow(equation.profile['Fz'].F[2]/kbar/gammabar, origin='bottom',
           extent=(np.amin(z)-dz/2, np.amax(z)+dz/2,
                   np.amin(v)-dv/2, np.amax(v)+dv/2), aspect='auto')
cb1 = plt.colorbar()
cb1.set_label('$F/(\hbar k \Gamma)$')
ax.set_xlabel('$x/(\hbar \Gamma/\mu_B B\')$')
ax.set_ylabel('$v/(\Gamma/k)$')
fig.subplots_adjust(left=0.12,right=0.9)

for sol in sols:
    ax.plot(sol.r[2], sol.v[2], 'w-', linewidth=0.375)

In [ ]:
# sols = []
# for v0 in v0s:
#     equation.set_initial_position_and_velocity(d, -1*v0*d/np.linalg.norm(d))
# #    equation.set_initial_position_and_velocity(np.array([0,0,z[0]]), np.array([0,0,v0]))
# #    equation.set_initial_pop(np.array([1.,0,0,0]))
#     equation.evolve_motion([0., 10000.], events=[captured_condition, lost_condition], progress_bar=True)
#     sols.append(equation.sol)

In [ ]:
# Phase diagrams are only useful right now in certain axises
fig = plt.figure()
ax = fig.gca(projection='3d')

# sol.r[0] is x and so on
# You can use dir to debug
for sol in sols:
    ax.plot(sol.r[0], sol.r[1], sol.r[2], linewidth=0.375)

ax.set_xlabel('$x$')
ax.set_ylabel('$y$')  
ax.set_zlabel('$z$')

# Set axises
# lims = np.array([ax.get_xlim(), ax.get_ylim(), ax.get_zlim()])
# bottom = np.amin(lims[:,0])
# top = np.amax(lims[:,1])
bottom = -1.5
top = 1.5      

ax.set_xlim(bottom, top)
ax.set_ylim(bottom, top)
ax.set_zlim(bottom, top)

In [ ]:
# Phase diagrams are only useful right now in certain axises
fig, ax = plt.subplots(1, 3, figsize=(6.5, 2.75))

for sol in sols:
    ax[0].plot(sol.t, sol.r[0], linewidth=0.375)
    ax[1].plot(sol.t, sol.r[1], linewidth=0.375)
    ax[2].plot(sol.t, sol.r[2], linewidth=0.375)

In [ ]:
# Phase diagrams are only useful right now in certain axises
fig, ax = plt.subplots(1, 3, figsize=(6.5, 2.75))

for sol in sols:
    ax[0].plot(sol.t, sol.v[0], linewidth=0.375)
    ax[1].plot(sol.t, sol.v[1], linewidth=0.375)
    ax[2].plot(sol.t, sol.v[2], linewidth=0.375)

By having two conditions, we can tell if the atom was lost or captured:

In [ ]:
for sol in sols:
    if len(sol.t_events[0]) == 1:
        print('captured')
    elif len(sol.t_events[1]) == 1:
        print('lost')

### Now, let's get even more fancy:

Let's define a function that figures out if we were captured or not, then use that to find the capture velocity:

In [ ]:
def iscaptured(v0, r0, eqn, captured_condition, lost_condition, nan_found, tmax=1000, **kwargs):
    eqn.set_initial_position_and_velocity(r0, -1*v0*r0/np.linalg.norm(r0))
    eqn.evolve_motion([0., tmax], events=[captured_condition, lost_condition, nan_found],
                      **kwargs)
    
    return len(eqn.sol.t_events[0]) == 1

iscaptured(0.1, d, equation, captured_condition, lost_condition, nan_found, tmax=1000)

See if we can find out where it changes:

In [ ]:
start_time = time.time()

bisectFindChangeValue(iscaptured, 0.1,
                      args=(d, equation, captured_condition, lost_condition, nan_found),
                      kwargs={'tmax':10000},
                      tol=1e-2
                     )

print("--- %s seconds ---" % (time.time() - start_time))

### Let's run the detuning and intensity:

We will figure out how the capture velocity depends on and compare to this equation from the paper in the introduction:

$$
v_c = \left(\frac{a_0^2\beta^2\kappa}{(1+\beta)^{3/2}}\right)^{1/3}\left(\frac{8\pi\delta^2}{1+\beta+4\delta^2}\right)^{1/3}\zeta^{-2/3}
$$

where $a_0 = \hbar k \Gamma/(2 m)$, $\zeta = \mu_B B'/(\hbar\Gamma)$, and $\kappa = 2\pi/(\lambda \Gamma)=k/\Gamma$ .  To compare, we need to express it in a way which connects with our formulae above.  The first thing to note is that $\zeta = 1/x_0$.  We also need to multiple both sides by $k/\Gamma$, so that we have $v_c/(\Gamma/k)$ on the left side, which is our observable.  Then, we realize that

$$
\frac{\hbar k\Gamma}{2m} = \frac{1}{2\bar{m}}\frac{x_0}{t_0^2}~~~~~\text{and}~~~~~\frac{k}{\Gamma} = \frac{t_0}{x_0} 
$$

Putting it all together:

$$
\frac{v_c}{\Gamma/k} = \frac{t_0}{x_0}\left(\frac{1}{2\bar{m}}\right)^{2/3} \frac{x_0^{2/3}}{t_0^{4/3}}\frac{t_0^{1/3}}{x_0^{1/3}} x_0^{2/3}\left(\frac{\beta^2}{(1+\beta)^{3/2}}\right)^{1/3}\left(\frac{8\pi\delta^2}{1+\beta+4\delta^2}\right)^{1/3} = \left(\frac{1}{2\bar{m}}\right)^{2/3}\left(\frac{\beta^2}{(1+\beta)^{3/2}}\right)^{1/3}\left(\frac{8\pi\delta^2}{1+\beta+4\delta^2}\right)^{1/3}
$$

In [ ]:
dets = -gammabar*np.logspace(-0.6, np.log10(3), 10)[::-1]
betas = np.array([0.5,1.,2.])

# dets = np.array([-1.5*gammabar])
# betas = np.array([1.0])

DETS, BETAS = np.meshgrid(dets, betas)

it = np.nditer([DETS, BETAS, None, None])

progress = progressBar()
for (det, beta, vc, iterations) in it:
    #beam_to_sim = pylcp.infinitePlaneWaveBeam
    #beam_to_sim = pylcp.gaussianBeam
    beam_to_sim = pylcp.clippedGaussianBeam

    #MOT_to_sim = conventional3DMOTBeams
    MOT_to_sim = aPHIMOT

    #MOT_to_sim_kwargs = {'rotation_angles':[np.pi/4, 0., 0.]} # Extra arguments for conventional3DMOTBeams
    MOT_to_sim_kwargs = {} # Extra arguments for aPHIMOT

    #laser_kwargs = {}
    #laser_kwargs = {'wb':wb} # Extra arguments for GaussianBeam
    laser_kwargs = {'wb':1000*wb, 'rs':wb} # Extra arguments for clippedGaussianBeam

    laserBeams = MOT_to_sim(beta=beta, delta=det, beam_type=beam_to_sim, k=kbar, **laser_kwargs, **MOT_to_sim_kwargs)
    
    if (np.sum(np.isnan(laserBeams.pol()), axis=(0, 1)) > 0):
        del laserBeams
        laserBeams = MOT_to_sim(beta=beta, delta=det, beam_type=beam_to_sim, k=kbar, **laser_kwargs, **MOT_to_sim_kwargs)
    
    equation = pylcp.heuristiceq(laserBeams, magField, gamma=gammabar, mass=mbar)

#     Hg, mugq = pylcp.hamiltonians.singleF(F=0, muB=1)
#     He, mueq = pylcp.hamiltonians.singleF(F=1, muB=1)

#     dq = pylcp.hamiltonians.dqij_two_bare_hyperfine(0, 1)

#     hamiltonian = pylcp.hamiltonian(mass=mbar)
#     hamiltonian.add_H_0_block('g', Hg)
#     hamiltonian.add_H_0_block('e', He)
#     hamiltonian.add_mu_q_block('g', mugq)
#     hamiltonian.add_mu_q_block('e', mueq)
#     hamiltonian.add_d_q_block('g','e', dq, gamma=gammabar, k=kbar)

#     equation = pylcp.rateeq(laserBeams, magField, hamiltonian)
    
    vc[...], iterations[...] = bisectFindChangeValue(
        iscaptured, 0.1,
        args=(d, equation, captured_condition, lost_condition, nan_found),
        kwargs={'tmax':10000, 'max_step':1.},
        tol=1e-2, debug=False
    )

    progress.update((it.iterindex+1)/it.itersize)

In [ ]:
def vc_from_paper(delta, beta, mbar):
    return 1/(2*mbar)**(2./3.)*(beta**2/(1+beta)**(3./2.))**(1./3.)*(8*np.pi*delta**2/(1+beta+4*delta**2))**(1./3.)

In [ ]:
fig, ax = plt.subplots(1, 1)
for (beta, vc_vs_det) in zip(betas, it.operands[2]):
    ax.plot(dets, vc_vs_det, label='$\\beta=%.1f$' % beta)
ax.legend(fontsize=8)
ax.set_xlabel('$\Delta/\Gamma$')
ax.set_ylabel('$v_c/(\Gamma/k)$')